# install

In [1]:
# Cài đặt hoặc nâng cấp vnstock
!pip install -U vnstock

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.8/275.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 3.0 MB/s eta 0:00:00


In [2]:
from vnstock import Quote
quote = Quote(symbol='ACB', source='KBS')


📋 Connecting Google Drive account
to save project settings.

Mounted at /content/drive


Ngày 8 — Deflated Sharpe Ratio (Bailey & López de Prado, 2014)

Mục tiêu:
1. Implement DSR đầy đủ (điều chỉnh skewness/kurtosis + số lượng trial N)
2. Áp dụng cho 12 factor đã build ở Ngày 4-5, trên universe 10 mã gốc — đặc biệt AlphaC_TrendCond
3. So sánh Sharpe thô vs t-stat (Sharpe/SE, chưa chỉnh N trials) vs DSR (đã chỉnh N=12 trials)

Câu hỏi cốt lõi: nếu tính đúng DSR ngay từ đầu (trên 10 mã), liệu nó có cảnh báo trước
việc AlphaC sẽ đảo dấu khi mở rộng lên 30 mã hay không — hay đây là lỗi mà chỉ multiple-testing
correction (không phải bản thân DSR) mới bắt được?

# Cell 1: Load dữ liệu

In [3]:
### Cell 1: Load dữ liệu (price + open + volume), universe 10 mã gốc
from vnstock import Quote
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, norm

SYMBOLS = ["BID", "ACB", "VCB", "VNM", "MSN", "MWG", "HPG", "GAS", "SSI", "VRE"]
START, END = "2024-01-01", "2025-12-31"
N_FWD = 5  # forward return horizon dùng để đánh giá factor

raw = {}
for sym in SYMBOLS:
    df = Quote(symbol=sym, source="KBS").history(start=START, end=END, interval="d")
    raw[sym] = df.set_index("time")[["close", "open", "volume"]]

price_df = pd.DataFrame({sym: d["close"] for sym, d in raw.items()}).sort_index()
open_df = pd.DataFrame({sym: d["open"] for sym, d in raw.items()}).sort_index()
volume_df = pd.DataFrame({sym: d["volume"] for sym, d in raw.items()}).sort_index()

price_df.head()

,BID,ACB,VCB,VNM,MSN,MWG,HPG,GAS,SSI,VRE
time,,,,,,,,,,
2024-01-02 07:00:00,32.53,14.80,55.04,57.89,68.4,40.88,18.55,64.82,18.06,22.31
2024-01-03 07:00:00,33.13,15.13,55.70,58.49,68.9,41.60,18.78,65.17,18.31,22.45
2024-01-04 07:00:00,33.02,15.32,56.63,58.49,68.1,41.60,18.75,65.77,18.67,22.60
2024-01-05 07:00:00,33.66,15.41,56.82,58.32,67.9,42.23,18.78,66.19,18.97,22.55
2024-01-08 07:00:00,35.10,15.34,57.22,57.81,66.6,41.60,18.82,65.85,18.95,22.89


# Cell 2: các hàm


In [4]:
### Cell 2: Helper functions dùng để build factor
def rank(df: pd.DataFrame) -> pd.DataFrame:
    return df.rank(axis=1, pct=True)

def delta(df: pd.DataFrame, d: int) -> pd.DataFrame:
    return df.diff(d)

def delay(df: pd.DataFrame, d: int) -> pd.DataFrame:
    return df.shift(d)

def ts_sum(df: pd.DataFrame, d: int) -> pd.DataFrame:
    return df.rolling(d).sum()

def ts_min(df: pd.DataFrame, d: int) -> pd.DataFrame:
    return df.rolling(d).min()

def ts_max(df: pd.DataFrame, d: int) -> pd.DataFrame:
    return df.rolling(d).max()

def ts_rank(df: pd.DataFrame, d: int) -> pd.DataFrame:
    return df.rolling(d).apply(lambda x: pd.Series(x).rank(pct=True).iloc[-1], raw=False)

def correlation(df1: pd.DataFrame, df2: pd.DataFrame, d: int) -> pd.DataFrame:
    return df1.rolling(d).corr(df2)

def zlema(series: pd.Series, period: int = 20) -> pd.Series:
    lag = (period - 1) // 2
    adjusted = series + (series - series.shift(lag))
    return adjusted.ewm(span=period, adjust=False).mean()

def calc_rsi(price: pd.DataFrame, window: int = 14) -> pd.DataFrame:
    delta_p = price.diff()
    gain = delta_p.clip(lower=0)
    loss = -delta_p.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / window, min_periods=window, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / window, min_periods=window, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - 100 / (1 + rs)

# Cell 3: Các factor ngày 4

In [5]:
### Cell 3: 7 factor gốc (Ngày 4)
returns = price_df.pct_change(fill_method=None)
adv20 = volume_df.rolling(20).mean()

# 1. Momentum 12-1: return 12 tháng, bỏ tháng gần nhất
mom_12_1 = price_df.shift(21) / price_df.shift(252) - 1

# 2. Mean-reversion: z-score return 5 ngày so với phân phối 20 phiên gần nhất
ret_5d = price_df.pct_change(5, fill_method=None)
reversion_z = (ret_5d - ret_5d.rolling(20).mean()) / ret_5d.rolling(20).std()

# 3. Volume ratio: volume hôm nay / volume trung bình 20 phiên
volume_ratio = volume_df / adv20

# 4. Alpha A: đảo dấu momentum 3 ngày, nhân đồng biến open-volume 10 ngày
alpha_a = (-1 * rank(delta(returns, 3))) * correlation(open_df, volume_df, 10)

# 5. Alpha B: vị trí giá 10 phiên, độ cong giá, bất thường volume 5 phiên
alpha_b = ((-1 * rank(ts_rank(price_df, 10)))
           * rank(delta(delta(price_df, 1), 1))
           * rank(ts_rank(volume_df / adv20, 5)))

# 6. Alpha C: ternary theo chế độ thị trường (trend rõ vs sideway)
trend_cond = (delta(ts_sum(price_df, 100) / 100, 100) / delay(price_df, 100)) <= 0.05
alpha_c = pd.DataFrame(
    np.where(trend_cond, -1 * (price_df - ts_min(price_df, 100)), -1 * delta(price_df, 3)),
    index=price_df.index, columns=price_df.columns,
)

# 7. ZLEMA mean-reversion: giá lệch khỏi ZLEMA-20 bao nhiêu %
zlema_df = price_df.apply(zlema)
alpha_zlema = -(price_df - zlema_df) / zlema_df

factors = {
    "Momentum_12_1": mom_12_1,
    "MeanReversion_Z": reversion_z,
    "VolumeRatio": volume_ratio,
    "AlphaA_RetCorr": alpha_a,
    "AlphaB_TSRankVol": alpha_b,
    "AlphaC_TrendCond": alpha_c,
    "ZLEMA_Reversion": alpha_zlema,
}

# Cell 4: Các factor ngày 5

In [6]:
### Cell 4: RSI-regime factors + ensemble với Alpha C (Ngày 5)
SLOPE_WINDOW = 5

rsi_df = calc_rsi(price_df, window=14)
rsi_slope = (rsi_df - rsi_df.shift(SLOPE_WINDOW)) / SLOPE_WINDOW

# AlphaD (No Regime): pure trend-following theo slope RSI
alpha_d = pd.DataFrame(np.sign(rsi_slope), index=rsi_df.index, columns=rsi_df.columns)

# AlphaD4 (Regime Switch): override reversal khi RSI chạm cực trị 100 ngày
rsi_high_100 = rsi_df >= ts_max(rsi_df, 100)
rsi_low_100 = rsi_df <= ts_min(rsi_df, 100)
alpha_d4 = pd.DataFrame(
    np.select([rsi_high_100.values, rsi_low_100.values], [-1, 1], default=alpha_d.values),
    index=rsi_df.index, columns=rsi_df.columns,
)

# AlphaD5 (Soft Regime): blend liên tục theo cường độ trend + percentile RSI
trend_strength = (delta(ts_sum(price_df, 100) / 100, 100) / delay(price_df, 100)).abs()
trend_weight = np.tanh(trend_strength / 0.05)

rsi_range_100 = ts_max(rsi_df, 100) - ts_min(rsi_df, 100)
rsi_pct_pos = (rsi_df - ts_min(rsi_df, 100)) / rsi_range_100
extreme_high = rsi_pct_pos >= 0.9
extreme_low = rsi_pct_pos <= 0.1

slope_signal_continuous = np.tanh(rsi_slope / 2)
blended_signal = trend_weight * slope_signal_continuous
alpha_d5 = pd.DataFrame(
    np.select([extreme_high.values, extreme_low.values], [-1, 1], default=blended_signal.values),
    index=rsi_df.index, columns=rsi_df.columns,
)

# Ensemble: Alpha C (rank-scaled [-1,1]) trung bình 50/50 với D4 / D5
alpha_c_scaled = 2 * rank(alpha_c) - 1
ensemble_c_d4 = (alpha_c_scaled + alpha_d4) / 2
ensemble_c_d5 = (alpha_c_scaled + alpha_d5) / 2

factors.update({
    "AlphaD_NoRegime": alpha_d,
    "AlphaD4_RegimeSwitch": alpha_d4,
    "AlphaD5_SoftRegime": alpha_d5,
    "Ensemble_C_D4": ensemble_c_d4,
    "Ensemble_C_D5": ensemble_c_d5,
})

print(f"Tổng số factor: {len(factors)}")
list(factors.keys())

Tổng số factor: 12


['Momentum_12_1',
 'MeanReversion_Z',
 'VolumeRatio',
 'AlphaA_RetCorr',
 'AlphaB_TSRankVol',
 'AlphaC_TrendCond',
 'ZLEMA_Reversion',
 'AlphaD_NoRegime',
 'AlphaD4_RegimeSwitch',
 'AlphaD5_SoftRegime',
 'Ensemble_C_D4',
 'Ensemble_C_D5']

# Cell 5: IC cho các factor

In [7]:
### Cell 5: Tính IC (Spearman, cross-sectional theo ngày) cho toàn bộ factor
fwd_ret = price_df.shift(-N_FWD) / price_df - 1

def calc_ic(factor_df: pd.DataFrame, fwd_ret_df: pd.DataFrame, min_valid: int = 5) -> pd.Series:
    """Cross-sectional Spearman IC giữa factor và forward return, theo từng ngày."""
    ic = {}
    for date in factor_df.index:
        f, r = factor_df.loc[date], fwd_ret_df.loc[date]
        valid = f.notna() & r.notna()
        if valid.sum() < min_valid:
            continue
        f_valid, r_valid = f[valid], r[valid]
        if f_valid.nunique() < 2 or r_valid.nunique() < 2:
            continue  # tránh warning/NaN khi factor hoặc forward return constant trong ngày đó
        ic[date] = spearmanr(f_valid, r_valid)[0]
    return pd.Series(ic).dropna()

ic_series = {name: calc_ic(f, fwd_ret) for name, f in factors.items()}

# Align về cùng tập ngày quan sát (giao của index) — công bằng khi so sánh giữa các factor
common_idx = None
for s in ic_series.values():
    common_idx = s.index if common_idx is None else common_idx.intersection(s.index)
ic_common = {name: s.loc[common_idx] for name, s in ic_series.items()}

def ic_stats(s: pd.Series) -> dict:
    return {
        "Mean IC": s.mean(), "Std IC": s.std(),
        "IC IR": s.mean() / s.std(),
        "t-stat": s.mean() / s.std() * np.sqrt(len(s)),
        "Hit Rate": (s > 0).mean(), "N Obs": len(s),
    }

summary = pd.DataFrame({name: ic_stats(s) for name, s in ic_common.items()}).T.round(4)
print(f"Common obs: {len(common_idx)}")
summary

Common obs: 201


,Mean IC,Std IC,IC IR,t-stat,Hit Rate,N Obs
Momentum_12_1,-0.0769,0.3931,-0.1955,-2.7721,0.4080,201.0
MeanReversion_Z,0.0007,0.3561,0.0021,0.0293,0.4876,201.0
VolumeRatio,0.0283,0.3412,0.0831,1.1776,0.5224,201.0
AlphaA_RetCorr,0.0372,0.3347,0.1112,1.5762,0.5274,201.0
AlphaB_TSRankVol,-0.0070,0.3595,-0.0194,-0.2750,0.5025,201.0
AlphaC_TrendCond,0.0734,0.3966,0.1851,2.6241,0.5821,201.0
ZLEMA_Reversion,-0.0172,0.3598,-0.0478,-0.6781,0.5075,201.0
AlphaD_NoRegime,0.0141,0.3518,0.0400,0.5669,0.5025,201.0
AlphaD4_RegimeSwitch,0.0247,0.3530,0.0699,0.9907,0.5124,201.0
AlphaD5_SoftRegime,-0.0154,0.3250,-0.0474,-0.6722,0.4627,201.0


# Cell 6: DSR


Deflated Sharpe Ratio (Bailey & López de Prado, 2014)

Coi chuỗi IC hàng ngày của mỗi factor như "return" của một chiến lược (đúng cách IC IR đã
được dùng như Sharpe ratio của tín hiệu xuyên suốt project). DSR trả lời: "xác suất Sharpe
THẬT lớn hơn mức Sharpe cao nhất kỳ vọng đạt được thuần túy do may rủi, nếu đã thử N chiến
lược độc lập" — gộp 2 điều chỉnh:

- **Non-normality**: skewness/kurtosis thật của chuỗi IC (không giả định normal).
- **Multiple testing**: benchmark so sánh là `SR_0*` (Sharpe max kỳ vọng dưới null với N trials),
  không phải 0.

In [8]:
### Cell 6: Deflated Sharpe Ratio — implementation đầy đủ
EULER_MASCHERONI = 0.5772156649015329

def sharpe_std_error(sr: float, skew: float, kurt: float, n_obs: int) -> float:
    """SE(SR_hat), điều chỉnh skew/kurtosis (Bailey & López de Prado, 2012).
    SE(SR) = sqrt( (1 - skew*SR + (kurt-1)/4 * SR^2) / (T-1) )
    kurt: kurtosis kiểu Pearson (normal=3), KHÔNG phải excess kurtosis.
    """
    return np.sqrt((1 - skew * sr + (kurt - 1) / 4 * sr ** 2) / (n_obs - 1))


def expected_max_sharpe(sr_trials: np.ndarray, n_trials: int) -> float:
    """SR_0*: Sharpe tối đa kỳ vọng thuần túy do may rủi nếu thử n_trials chiến lược độc lập,
    không chiến lược nào có skill thật (null hypothesis).
    SR_0* = sqrt(Var[SR_n]) * [(1-γ)*Z^-1(1-1/N) + γ*Z^-1(1-1/(N*e))]
    """
    var_sr = np.var(sr_trials, ddof=1)
    sr_std = np.sqrt(var_sr)
    return sr_std * (
        (1 - EULER_MASCHERONI) * norm.ppf(1 - 1 / n_trials)
        + EULER_MASCHERONI * norm.ppf(1 - 1 / (n_trials * np.e))
    )


def probabilistic_sharpe_ratio(returns: pd.Series, sr_benchmark: float = 0.0) -> dict:
    """PSR(SR*): xác suất Sharpe thật > benchmark, chỉnh skew/kurtosis, CHƯA chỉnh multiple testing."""
    n_obs = len(returns)
    sr_hat = returns.mean() / returns.std()
    skew = returns.skew()
    kurt = returns.kurtosis() + 3  # pandas .kurtosis() = excess kurtosis -> +3 về Pearson kurtosis

    se_sr = sharpe_std_error(sr_hat, skew, kurt, n_obs)
    z = (sr_hat - sr_benchmark) / se_sr
    return {"SR_hat": sr_hat, "Skewness": skew, "Kurtosis": kurt,
            "SE(SR_hat)": se_sr, "Z-score": z, "PSR": norm.cdf(z), "N_obs": n_obs}


def deflated_sharpe_ratio(returns: pd.Series, sr_trials: np.ndarray, n_trials: int) -> dict:
    """DSR = PSR(SR_0*): benchmark là SR_0* thay vì 0.
    DSR ~ 1   -> Sharpe vượt xa mức may rủi kỳ vọng trong N lần thử -> skill đáng tin.
    DSR ~ 0.5 -> không phân biệt được với kết quả ngẫu nhiên tốt nhất trong N lần thử.
    """
    n_obs = len(returns)
    sr_hat = returns.mean() / returns.std()
    skew = returns.skew()
    kurt = returns.kurtosis() + 3

    sr_0 = expected_max_sharpe(sr_trials, n_trials)
    se_sr = sharpe_std_error(sr_hat, skew, kurt, n_obs)
    z = (sr_hat - sr_0) / se_sr

    return {"SR_hat": sr_hat, "Skewness": skew, "Kurtosis": kurt,
            "SR_0_star": sr_0, "SE(SR_hat)": se_sr, "Z-score": z,
            "DSR": norm.cdf(z), "N_obs": n_obs, "N_trials": n_trials}

# Cell 7: Check SR0

In [9]:
### Cell 9: Sanity check — DSR trên chuỗi random thuần (kỳ vọng DSR ~ 0.5, không có skill thật)
rng = np.random.default_rng(0)
fake_returns = pd.Series(rng.normal(0, 1, 250))
fake_trials = rng.normal(0, 0.15, 12)

print(deflated_sharpe_ratio(fake_returns, sr_trials=fake_trials, n_trials=12))

{'SR_hat': np.float64(-0.00580996468069035), 'Skewness': np.float64(-0.018074045091663046), 'Kurtosis': np.float64(2.9262689551813943), 'SR_0_star': np.float64(0.24391482343655813), 'SE(SR_hat)': np.float64(0.06336961271417571), 'Z-score': np.float64(-3.940765572351136), 'DSR': np.float64(4.061098667464669e-05), 'N_obs': 250, 'N_trials': 12}


# Cell 8: DSR cho các factor

In [10]:
### Cell 8: DSR cho toàn bộ 12 factor, N_trials=12 (đúng số factor đã test trong project)
N_TRIALS = 12
sr_trials = summary["IC IR"].values  # dùng Sharpe (IC IR) của cả 12 factor để ước lượng Var[SR_n]

dsr_results = {
    name: deflated_sharpe_ratio(ic_s, sr_trials=sr_trials, n_trials=N_TRIALS)
    for name, ic_s in ic_common.items()
}
dsr_table = pd.DataFrame(dsr_results).T.round(4)
dsr_table.sort_values("DSR", ascending=False)

,SR_hat,Skewness,Kurtosis,SR_0_star,SE(SR_hat),Z-score,DSR,N_obs,N_trials
Ensemble_C_D4,0.2182,0.2097,2.5286,0.1952,0.0697,0.3300,0.6293,201.0,12.0
AlphaC_TrendCond,0.1851,-0.1652,2.2684,0.1952,0.0722,-0.1403,0.4442,201.0,12.0
Ensemble_C_D5,0.1642,0.0618,2.3015,0.1952,0.0707,-0.4395,0.3302,201.0,12.0
AlphaA_RetCorr,0.1112,0.1042,2.4553,0.1952,0.0705,-1.1927,0.1165,201.0,12.0
VolumeRatio,0.0831,-0.0675,2.4735,0.1952,0.0710,-1.5796,0.0571,201.0,12.0
AlphaD4_RegimeSwitch,0.0699,-0.0496,2.1896,0.1952,0.0709,-1.7682,0.0385,201.0,12.0
AlphaD_NoRegime,0.0400,0.0481,2.3532,0.1952,0.0707,-2.1968,0.0140,201.0,12.0
MeanReversion_Z,0.0021,0.1269,2.5308,0.1952,0.0707,-2.7319,0.0031,201.0,12.0
AlphaB_TSRankVol,-0.0194,0.0194,2.3495,0.1952,0.0707,-3.0343,0.0012,201.0,12.0
ZLEMA_Reversion,-0.0478,-0.1191,2.6711,0.1952,0.0705,-3.4453,0.0003,201.0,12.0


Chiến lược Ensemble alpha C và alpha D4 đem lại DSR cao nhất trên bảng dữ liệu 10 mã

gần như đồng thuận với mean IC (kết quả của chiến lược này cao thứ 2)

# Cell 9: Sharpe thô vs t-stat vs DSR

In [11]:
### Cell 9: Sharpe thô vs t-stat (Sharpe/SE, chưa chỉnh N) vs DSR (chỉnh N=12 trials)
compare_3way = pd.DataFrame({
    "Sharpe_tho (IC_IR)": summary["IC IR"],
    "t_stat (chua chinh N trials)": summary["t-stat"],
    "DSR_Zscore (N=12)": dsr_table["Z-score"],
    "DSR (0-1)": dsr_table["DSR"],
}).round(4).sort_values("DSR (0-1)", ascending=False)

compare_3way

,Sharpe_tho (IC_IR),t_stat (chua chinh N trials),DSR_Zscore (N=12),DSR (0-1)
Ensemble_C_D4,0.2182,3.0938,0.3300,0.6293
AlphaC_TrendCond,0.1851,2.6241,-0.1403,0.4442
Ensemble_C_D5,0.1642,2.3273,-0.4395,0.3302
AlphaA_RetCorr,0.1112,1.5762,-1.1927,0.1165
VolumeRatio,0.0831,1.1776,-1.5796,0.0571
AlphaD4_RegimeSwitch,0.0699,0.9907,-1.7682,0.0385
AlphaD_NoRegime,0.0400,0.5669,-2.1968,0.0140
MeanReversion_Z,0.0021,0.0293,-2.7319,0.0031
AlphaB_TSRankVol,-0.0194,-0.2750,-3.0343,0.0012
ZLEMA_Reversion,-0.0478,-0.6781,-3.4453,0.0003


Cả 3 đều đồng thuận Ensemble giữa alpha C và D4 là hợp lý nhất

# Cell 10: Nhìn kỹ alpha C


In [12]:
### Cell 10: Zoom riêng AlphaC_TrendCond — nhân vật chính của câu chuyện "đảo dấu ở 30 mã"
print("=== AlphaC_TrendCond, universe 10 mã ===")
print(compare_3way.loc["AlphaC_TrendCond"])
print()
print(f"DSR median toàn bộ 12 factor: {dsr_table['DSR'].median():.4f}")
print(f"DSR AlphaC_TrendCond:         {dsr_table.loc['AlphaC_TrendCond', 'DSR']:.4f}")
print(f"Rank DSR của AlphaC trong {N_TRIALS} factor: "
      f"{(dsr_table['DSR'] > dsr_table.loc['AlphaC_TrendCond', 'DSR']).sum() + 1} / {N_TRIALS}")

=== AlphaC_TrendCond, universe 10 mã ===
Sharpe_tho (IC_IR)              0.1851
t_stat (chua chinh N trials)    2.6241
DSR_Zscore (N=12)              -0.1403
DSR (0-1)                       0.4442
Name: AlphaC_TrendCond, dtype: float64

DSR median toàn bộ 12 factor: 0.0262
DSR AlphaC_TrendCond:         0.4442
Rank DSR của AlphaC trong 12 factor: 2 / 12


Alpha C trong dữ liệu 10 mã chưa đứng đầu (thua chiến lược Ensemble và D4)

Tiến hành chạy lại trên 30 mã để xem kết luận có thay đổi hay đổi dấu không

Vì hiện tại vẫn chưa cho ra kết quả khả quan